In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 5a: Validasi dan overfitting

Cerita lengkap, perumpamaan, dan batas analogi ada di `README.md`. Hitung dulu, baru jalankan sel di bawah.

Empat nilai: 2, 4, 6, dan 8. Dua terakhir disisihkan sebagai data uji.

```
2    4    |    6    8
latih     |    uji
```

1. Berapa rata-rata keempatnya?
2. Berapa rata-rata dua yang pertama?
3. Rata-rata mana yang boleh dipakai saat model belajar?


In [2]:
# Pustaka adalah kumpulan fungsi siap pakai, bukan rak buku.
# Paket adalah bundel kode yang dipasang ke lingkungan proyek.
# NumPy adalah pustaka sekaligus paket untuk hitungan pada array.
# Array adalah kumpulan nilai sejenis dengan bentuk yang tetap, yang dihitung sekaligus.
import numpy as np


In [3]:
# Satu ide: simpan empat nilai. Dua terakhir adalah data uji.
nilai = np.array([2, 4, 6, 8])
nilai


array([2, 4, 6, 8])

In [4]:
# Satu ide: rata-rata seluruh nilai, termasuk data uji.
print("mean_semua", float(nilai.mean()))


mean_semua 5.0


In [5]:
# Satu ide: rata-rata dua nilai pertama, yaitu data latih saja.
print("mean_latih", float(nilai[:2].mean()))


mean_latih 3.0


## Data leakage

Data latih dipakai model untuk belajar. Data uji disisihkan dan tidak dilihat saat belajar, lalu dipakai untuk mengukur model pada data baru.

Rata-rata dari seluruh baris memakai angka yang seharusnya ditahan. Nama teknisnya **data leakage**: informasi yang seharusnya tidak tersedia saat model belajar ikut terpakai, sehingga skor terlihat lebih bagus dari kenyataan.

Contoh berikutnya tidak punya pola. Jawabannya diundi. Skor yang jujur berada di sekitar setengah.


In [6]:
# StandardScaler menghitung rata-rata kolom.
# fit menghitung angka itu dari baris yang diberikan.
# transform menerapkan angka itu ke baris lain.
# reshape menyusun ulang angka menjadi satu kolom.
# Empat nama ini bukan istilah baru.
from sklearn.preprocessing import StandardScaler

# Satu ide: rata-rata dari keempat angka, termasuk data uji.
nilai_kolom = nilai.reshape(-1, 1)
scaler_salah = StandardScaler()
scaler_salah.fit(nilai_kolom)
print("mean_scaler_salah", float(scaler_salah.mean_[0]))


mean_scaler_salah 5.0


In [7]:
# Satu ide: rata-rata hanya dari dua angka data latih.
scaler_benar = StandardScaler()
scaler_benar.fit(nilai_kolom[:2])
print("mean_scaler_benar", float(scaler_benar.mean_[0]))


mean_scaler_benar 3.0


In [8]:
# default_rng(SEED) membuat Generator NumPy. Seed berupa bilangan bulat membuat undian itu berulang.
# standard_normal mengundi sebaran normal baku: rata-rata 0 dan simpangan baku 1. (400, 2000) adalah bentuk tabel keluaran.
# integers(0, 2) mengundi bilangan bulat dari 0 sampai sebelum 2, jadi hanya 0 atau 1.
# Satu ide: buat tabel acak. Jawaban juga acak, jadi tidak ada pola sungguhan.
rng = np.random.default_rng(SEED)
X_acak = rng.standard_normal((400, 2000))
y_acak = rng.integers(0, 2, size=400)
print("baris_acak", int(X_acak.shape[0]))
print("kolom_acak", int(X_acak.shape[1]))


baris_acak 400
kolom_acak 2000


In [ ]:
# Satu ide: siapkan alat memilih kolom dan membagi baris.
# SelectKBest memilih kolom. f_classif mengukur beda rata-rata kolom antara dua kelas.
# k adalah banyaknya kolom yang dipilih.
# train_test_split membagi baris menjadi data latih dan data uji.
# test_size=0.25 menyisihkan seperempat baris sebagai data uji.
# LogisticRegression menyusun batas dua golongan dari jumlah berbobot.
# max_iter membatasi berapa kali hitungan model diulang.
# Nama-nama ini bukan istilah baru.
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [9]:
# Satu ide: cara salah. Kolom dipilih dari seluruh baris, termasuk data uji.
pilih_salah = SelectKBest(f_classif, k=15)
X_terpilih = pilih_salah.fit_transform(X_acak, y_acak)
X_latih_s, X_uji_s, y_latih_s, y_uji_s = train_test_split(
    X_terpilih, y_acak, test_size=0.25, random_state=SEED
)
print("kolom_terpilih", int(X_terpilih.shape[1]))


kolom_terpilih 15


In [10]:
# Satu ide: skor cara yang salah.
# Skor adalah bagian tebakan yang sama dengan jawaban. Makin tinggi, makin sering cocok.
# Ini bukan istilah baru. rata-rata selisih mutlak dari modul sebelumnya mengukur jarak angka, dan makin kecil makin dekat.
model_salah = LogisticRegression(max_iter=500, random_state=SEED)
model_salah.fit(X_latih_s, y_latih_s)
print("skor_salah", float(model_salah.score(X_uji_s, y_uji_s)))


skor_salah 0.74


In [11]:
# Satu ide: cara benar. Bagi dulu, lalu pilih kolom hanya dari data latih.
X_latih_b, X_uji_b, y_latih_b, y_uji_b = train_test_split(
    X_acak, y_acak, test_size=0.25, random_state=SEED
)
pilih_benar = SelectKBest(f_classif, k=15)
X_latih_bp = pilih_benar.fit_transform(X_latih_b, y_latih_b)
X_uji_bp = pilih_benar.transform(X_uji_b)
print("baris_latih", int(X_latih_b.shape[0]))
print("baris_uji", int(X_uji_b.shape[0]))


baris_latih 300
baris_uji 100


In [12]:
# Satu ide: skor cara yang benar.
model_benar = LogisticRegression(max_iter=500, random_state=SEED)
model_benar.fit(X_latih_bp, y_latih_b)
print("skor_benar", float(model_benar.score(X_uji_bp, y_uji_b)))


skor_benar 0.47


## Catatan jam

Jam di bawah berurutan. Mengocok berarti mengacak urutan baris sebelum dibagi. Mengocok membuat sebagian jam masa depan masuk data latih.

TimeSeriesSplit memotong data berurutan tanpa mengocok: jam pada data uji selalu di belakang jam pada data latih. Bukan istilah baru. `urut 1` berarti jam uji dimulai setelah jam latih berakhir.

DecisionTreeClassifier menyusun aturan jika-maka dengan memotong rentang angka. max_depth membatasi berapa kali aturan boleh bercabang. np.sin membuat gelombang berulang. Bukan istilah baru.

Pohon yang sudah melihat jam di sekitar titik uji bisa meniru gelombang di rentang itu. Pohon yang hanya melihat jam awal belum melihat nomor jam yang jauh lebih besar.


In [13]:
# Satu ide: hitung jam masa depan yang masuk data latih karena urutan dikocok.
jam = np.arange(80)
acak_latih, acak_uji = train_test_split(
    jam, test_size=0.25, random_state=SEED, shuffle=True
)
print("acak_uji_min", int(acak_uji.min()))
print("acak_latih_max", int(acak_latih.max()))
print("masa_depan_di_latih", int(np.sum(acak_latih > acak_uji.min())))


acak_uji_min 0
acak_latih_max 79
masa_depan_di_latih 60


In [14]:
# Satu ide: tiap giliran menahan jam yang lebih belakangan.
# Nomor giliran dimulai dari 0.
from sklearn.model_selection import TimeSeriesSplit
pembagi_waktu = TimeSeriesSplit(n_splits=3)
for i, (bagian_latih, bagian_uji) in enumerate(pembagi_waktu.split(jam)):
    akhir = int(bagian_latih.max())
    awal = int(bagian_uji.min())
    print("giliran", i, "latih_akhir", akhir, "uji_awal", awal, "urut", int(akhir < awal))


giliran 0 latih_akhir 19 uji_awal 20 urut 1
giliran 1 latih_akhir 39 uji_awal 40 urut 1
giliran 2 latih_akhir 59 uji_awal 60 urut 1


In [15]:
# Satu ide: pola hujan dari gelombang pada nomor jam.
nomor = np.arange(240)
y_jam = (np.sin(nomor / 4.0) > 0).astype(int)
X_jam = nomor.reshape(-1, 1).astype(float)
print("baris_jam", int(len(nomor)))


baris_jam 240


In [16]:
# Satu ide: skor bila jam dikocok sebelum dibagi.
# Kedalaman 6 cukup untuk memotong gelombang di jam yang sudah dilihat.
from sklearn.tree import DecisionTreeClassifier

Xk_l, Xk_u, yk_l, yk_u = train_test_split(
    X_jam, y_jam, test_size=0.25, random_state=SEED, shuffle=True
)
pohon_kocok = DecisionTreeClassifier(max_depth=6, random_state=SEED)
pohon_kocok.fit(Xk_l, yk_l)
print("skor_kocok", float(pohon_kocok.score(Xk_u, yk_u)))


skor_kocok 0.6666666666666666


In [17]:
# Satu ide: skor bila data uji adalah jam yang benar-benar belakangan.
potong = int(0.75 * len(nomor))
pohon_urut = DecisionTreeClassifier(max_depth=6, random_state=SEED)
pohon_urut.fit(X_jam[:potong], y_jam[:potong])
print("skor_urut", float(pohon_urut.score(X_jam[potong:], y_jam[potong:])))


skor_urut 0.55


## Overfitting

Overfitting artinya model terlalu menyesuaikan diri dengan data latih, termasuk kebetulan-kebetulannya, sehingga hasilnya bagus di data latih tetapi buruk di data baru. Arti ini sama dengan modul regresi. Modul ini tidak menambah istilah baru untuk itu.

make_classification mengundi soal klasifikasi di memori, sesuai dokumentasi scikit-learn. Arti tiap parameternya ada di sel berikutnya. Tidak ada berkas yang diunduh. Bukan data asli perusahaan.

Pohon tanpa batas boleh terus bercabang sampai baris latih cocok. Pohon dengan `max_depth=2` hanya bercabang dua kali.


### Arti parameter

Menurut dokumentasi scikit-learn, make_classification mengundi soal klasifikasi.

n_samples adalah banyaknya baris.

n_features adalah banyaknya kolom semuanya: kolom informatif, kolom redundan, kolom salinan, dan sisanya kolom acak yang tidak berguna.

n_informative adalah banyaknya kolom informatif. Tiap kelas punya kelompok sebaran normal (simpangan baku 1) di sekitar sudut hypercube berdimensi sebanyak itu.

Hypercube adalah kotak bersudut di beberapa dimensi. Bukan istilah baru.

Di dalam tiap kelompok, kolom informatif diundi dari sebaran normal baku (rata-rata 0, simpangan baku 1), lalu digabung secara linear, sebelum kelompok diletakkan di sudut hypercube.

n_redundant adalah banyaknya kolom kombinasi linear acak dari kolom informatif.

n_repeated adalah banyaknya kolom salinan, diambil acak dari kolom informatif dan redundan.

n_clusters_per_class adalah banyaknya kelompok pada tiap kelas.

class_sep mengalikan ukuran hypercube. Panjang sisinya 2 kali class_sep. Makin besar, kelompok makin berjauhan dan soal makin mudah.

flip_y adalah bagian baris yang kelasnya diundi ulang secara acak.

random_state membuat undian pembuatan data ini berulang selama angkanya sama.

Nama parameter ini bukan istilah baru.


In [18]:
# Satu ide: data sintetis untuk membandingkan pohon dalam dan pohon pendek.
from sklearn.datasets import make_classification
X_pola, y_pola = make_classification(
    n_samples=400,
    n_features=30,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    class_sep=0.7,
    flip_y=0.1,
    random_state=SEED,
)
print("baris_pola", int(X_pola.shape[0]))
print("kolom_pola", int(X_pola.shape[1]))


baris_pola 400
kolom_pola 30


In [19]:
# Satu ide: sisihkan 30 persen baris sebagai data uji.
X_latih_pola, X_uji_pola, y_latih_pola, y_uji_pola = train_test_split(
    X_pola, y_pola, test_size=0.3, random_state=SEED, shuffle=True
)
print("n_latih", int(len(y_latih_pola)))
print("n_uji", int(len(y_uji_pola)))


n_latih 280
n_uji 120


In [20]:
# Satu ide: skor pohon tanpa batas pada data latih.
pohon_dalam = DecisionTreeClassifier(max_depth=None, random_state=SEED)
pohon_dalam.fit(X_latih_pola, y_latih_pola)
skor_latih_dalam = float(pohon_dalam.score(X_latih_pola, y_latih_pola))
print("skor_latih_dalam", skor_latih_dalam)


skor_latih_dalam 1.0


In [21]:
# Satu ide: skor pohon yang sama pada data uji.
skor_uji_dalam = float(pohon_dalam.score(X_uji_pola, y_uji_pola))
print("skor_uji_dalam", skor_uji_dalam)


skor_uji_dalam 0.6666666666666666


In [22]:
# Satu ide: skor pohon pendek pada data latih.
pohon_dangkal = DecisionTreeClassifier(max_depth=2, random_state=SEED)
pohon_dangkal.fit(X_latih_pola, y_latih_pola)
skor_latih_dangkal = float(pohon_dangkal.score(X_latih_pola, y_latih_pola))
print("skor_latih_dangkal", skor_latih_dangkal)


skor_latih_dangkal 0.8107142857142857


In [23]:
# Satu ide: skor pohon pendek pada data uji.
skor_uji_dangkal = float(pohon_dangkal.score(X_uji_pola, y_uji_pola))
print("skor_uji_dangkal", skor_uji_dangkal)


skor_uji_dangkal 0.8083333333333333


In [24]:
# Satu ide: selisih skor. Selisih besar berarti hasil di data latih jauh lebih baik daripada di data baru.
print("selisih_dalam", float(skor_latih_dalam - skor_uji_dalam))
print("selisih_dangkal", float(skor_latih_dangkal - skor_uji_dangkal))


selisih_dalam 0.33333333333333337
selisih_dangkal 0.0023809523809523725


## Cross-validation

```
giliran 0   [uji] [latih] [latih] [latih] [latih]
giliran 1   [latih] [uji] [latih] [latih] [latih]
```

**Cross-validation** mengukur model dengan memutar bagian yang disisihkan. Data dibagi beberapa lipatan. Tiap lipatan bergiliran menjadi data uji. Skor lipatan lalu dirata-rata.

Lipatan adalah satu potongan pada giliran itu. Kata ini bagian dari cross-validation, bukan istilah terpisah. Nomor giliran dimulai dari 0.

KFold memotong lipatan. cross_val_score mengulang pengukuran tiap giliran. Kita mengocok karena baris ini tidak berurutan waktu. Giliran hanya memakai data latih. Data uji yang sudah disisihkan tidak ikut. Empat kalimat ini bukan istilah baru.

Rumus, setelah gambar di atas: rata-rata = (jumlah skor lima lipatan) / 5. Angkanya dicetak sel berikutnya.


In [25]:
# Satu ide: skor tiap lipatan pada data latih saja.
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
pohon_giliran = DecisionTreeClassifier(max_depth=None, random_state=SEED)
skor_lipatan = cross_val_score(
    pohon_giliran, X_latih_pola, y_latih_pola, cv=cv
)
for i, skor in enumerate(skor_lipatan):
    print("lipatan", i, float(skor))


lipatan 0 0.6785714285714286
lipatan 1 0.7142857142857143
lipatan 2 0.7321428571428571
lipatan 3 0.5892857142857143
lipatan 4 0.6964285714285714


In [26]:
# Satu ide: rata-rata skor lipatan.
print("rata_lipatan", float(skor_lipatan.mean()))


rata_lipatan 0.6821428571428572
